In [1]:
import numpy as np
import pandas as pd

import torch
import torch.nn as nn

from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_absolute_error
from sklearn.metrics import mean_squared_error

from torch.utils.data import TensorDataset
from torch.utils.data import DataLoader

In [2]:
df = pd.read_hdf("pems-bay.h5")

traffic = df.values

print(traffic.shape)

(52116, 325)


In [3]:
scaler = MinMaxScaler()

traffic_scaled = scaler.fit_transform(
    traffic
)

In [4]:
X = []
y = []

sequence_length = 12

for i in range(
    len(traffic_scaled)
    - sequence_length
):

    X.append(
        traffic_scaled[
            i:i+sequence_length
        ]
    )

    y.append(
        traffic_scaled[
            i+sequence_length
        ]
    )

X = np.array(X)
y = np.array(y)

print(X.shape)
print(y.shape)

(52104, 12, 325)
(52104, 325)


In [5]:
split = int(
    len(X) * 0.8
)

X_train = X[:split]
X_test = X[split:]

y_train = y[:split]
y_test = y[split:]

In [6]:
X_train = torch.tensor(
    X_train,
    dtype=torch.float32
)

X_test = torch.tensor(
    X_test,
    dtype=torch.float32
)

y_train = torch.tensor(
    y_train,
    dtype=torch.float32
)

y_test = torch.tensor(
    y_test,
    dtype=torch.float32
)

In [7]:
train_dataset = TensorDataset(
    X_train,
    y_train
)

train_loader = DataLoader(
    train_dataset,
    batch_size=64,
    shuffle=True
)

In [8]:
class MultiScaleTemporalConv(nn.Module):

    def __init__(
        self,
        in_channels,
        out_channels
    ):

        super().__init__()

        self.conv3 = nn.Conv2d(
            in_channels,
            out_channels,
            kernel_size=(3,1),
            padding=(1,0)
        )

        self.conv5 = nn.Conv2d(
            in_channels,
            out_channels,
            kernel_size=(5,1),
            padding=(2,0)
        )

        self.conv7 = nn.Conv2d(
            in_channels,
            out_channels,
            kernel_size=(7,1),
            padding=(3,0)
        )

    def forward(self,x):

        out3 = self.conv3(x)

        out5 = self.conv5(x)

        out7 = self.conv7(x)

        return torch.relu(
            out3 + out5 + out7
        )

In [9]:
class AdaptiveGraphConv(nn.Module):

    def __init__(
        self,
        num_nodes,
        channels
    ):

        super().__init__()

        self.node_embeddings = nn.Parameter(
            torch.randn(
                num_nodes,
                16
            )
        )

        self.weight = nn.Linear(
            channels,
            channels
        )

    def forward(self,x):

        adaptive_adj = torch.softmax(
            torch.relu(
                self.node_embeddings
                @
                self.node_embeddings.T
            ),
            dim=1
        )

        x = torch.einsum(
            "ij,bctj->bcti",
            adaptive_adj,
            x
        )

        x = x.permute(
            0,2,3,1
        )

        x = self.weight(x)

        x = x.permute(
            0,3,1,2
        )

        return torch.relu(x)

In [10]:
class LearnableHypergraphConv(nn.Module):

    def __init__(
        self,
        num_nodes,
        num_clusters,
        channels
    ):

        super().__init__()

        self.node_embeddings = nn.Parameter(
            torch.randn(
                num_nodes,
                16
            )
        )

        self.cluster_embeddings = nn.Parameter(
            torch.randn(
                num_clusters,
                16
            )
        )

        self.weight = nn.Linear(
            channels,
            channels
        )

    def forward(self,x):

        H = torch.softmax(
            self.node_embeddings
            @
            self.cluster_embeddings.T,
            dim=1
        )

        hyper_adj = H @ H.T

        hyper_adj = hyper_adj / (
            hyper_adj.sum(
                dim=1,
                keepdim=True
            )
            + 1e-6
        )

        x = torch.einsum(
            "ij,bctj->bcti",
            hyper_adj,
            x
        )

        x = x.permute(
            0,2,3,1
        )

        x = self.weight(x)

        x = x.permute(
            0,3,1,2
        )

        return torch.relu(x)

In [11]:
class LCHASTGCN(nn.Module):

    def __init__(self):

        super().__init__()

        self.temp1 = MultiScaleTemporalConv(
            1,
            32
        )

        self.graph = AdaptiveGraphConv(
            num_nodes=325,
            channels=32
        )

        self.hypergraph = LearnableHypergraphConv(
            num_nodes=325,
            num_clusters=16,
            channels=32
        )

        self.temp2 = MultiScaleTemporalConv(
            32,
            32
        )

        self.fc = nn.Linear(
            32,
            1
        )

    def forward(self,x):

        x = x.unsqueeze(1)

        x = self.temp1(x)

        graph_features = self.graph(x)

        hyper_features = self.hypergraph(x)

        x = graph_features + hyper_features

        x = self.temp2(x)

        x = x.mean(dim=2)

        x = x.permute(
            0,
            2,
            1
        )

        x = self.fc(x)

        return x.squeeze(-1)

In [12]:
model = LCHASTGCN()

X_batch, y_batch = next(
    iter(train_loader)
)

pred = model(X_batch)

print(pred.shape)
print(y_batch.shape)

torch.Size([64, 325])
torch.Size([64, 325])


In [13]:
model = LCHASTGCN()

criterion = nn.MSELoss()

optimizer = torch.optim.Adam(
    model.parameters(),
    lr=0.001
)

epochs = 30

for epoch in range(epochs):

    model.train()

    total_loss = 0

    for X_batch, y_batch in train_loader:

        optimizer.zero_grad()

        pred = model(X_batch)

        loss = criterion(
            pred,
            y_batch
        )

        loss.backward()

        optimizer.step()

        total_loss += loss.item()

    print(
        f"Epoch {epoch+1}/{epochs} "
        f"Loss: {total_loss/len(train_loader):.6f}"
    )

Epoch 1/30 Loss: 0.007492
Epoch 2/30 Loss: 0.001001
Epoch 3/30 Loss: 0.000674
Epoch 4/30 Loss: 0.000593
Epoch 5/30 Loss: 0.000571
Epoch 6/30 Loss: 0.000558
Epoch 7/30 Loss: 0.000548
Epoch 8/30 Loss: 0.000536
Epoch 9/30 Loss: 0.000538
Epoch 10/30 Loss: 0.000520
Epoch 11/30 Loss: 0.000518
Epoch 12/30 Loss: 0.000505
Epoch 13/30 Loss: 0.000502
Epoch 14/30 Loss: 0.000501
Epoch 15/30 Loss: 0.000496
Epoch 16/30 Loss: 0.000492
Epoch 17/30 Loss: 0.000492
Epoch 18/30 Loss: 0.000488
Epoch 19/30 Loss: 0.000485
Epoch 20/30 Loss: 0.000487
Epoch 21/30 Loss: 0.000483
Epoch 22/30 Loss: 0.000481
Epoch 23/30 Loss: 0.000482
Epoch 24/30 Loss: 0.000482
Epoch 25/30 Loss: 0.000477
Epoch 26/30 Loss: 0.000478
Epoch 27/30 Loss: 0.000478
Epoch 28/30 Loss: 0.000476
Epoch 29/30 Loss: 0.000475
Epoch 30/30 Loss: 0.000476


In [14]:
test_dataset = TensorDataset(
    X_test,
    y_test
)

test_loader = DataLoader(
    test_dataset,
    batch_size=64,
    shuffle=False
)

all_predictions = []
all_targets = []

model.eval()

with torch.no_grad():

    for X_batch, y_batch in test_loader:

        pred = model(X_batch)

        all_predictions.append(
            pred.numpy()
        )

        all_targets.append(
            y_batch.numpy()
        )

predictions = np.concatenate(
    all_predictions,
    axis=0
)

true_values = np.concatenate(
    all_targets,
    axis=0
)

mae = mean_absolute_error(
    true_values,
    predictions
)

rmse = np.sqrt(
    mean_squared_error(
        true_values,
        predictions
    )
)

print("MAE:", mae)
print("RMSE:", rmse)

MAE: 0.012627707
RMSE: 0.023152435
